<a href="https://colab.research.google.com/github/Khushi-Dua/Be-Practical-Training/blob/main/Day1_Lab_RecordManager_starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import json
import os
import sys
import csv
from typing import Dict, Any

DATABASE_FILE = "sample_records.json"

STUDENT_REGISTRY: Dict[str, Dict[str, Any]] = {}


def load_records_from_json(file_path: str) -> Dict[str, Dict[str, Any]]:
    """Loads student records safely from a JSON file."""

    if not os.path.exists(file_path):
        print(f"[WARN] Database file '{file_path}' not found. Starting with empty registry.")
        return {}

    try:
        with open(file_path, "r", encoding="utf-8") as file:
            data = json.load(file)
            print(f"[SUCCESS] Loaded {len(data)} record(s) from {file_path}.")
            return data

    except json.JSONDecodeError as json_err:
        print(f"[ERROR] Corrupted JSON structure in '{file_path}': {json_err}")
        return {}

    except Exception as err:
        print(f"[UNEXPECTED ERROR] Failed to load data: {err}")
        return {}


def view_all_records(registry: Dict[str, Dict[str, Any]]) -> None:
    """Prints all student records in a formatted tabular view."""

    if not registry:
        print("\n[INFO] No records found in the registry.")
        return

    separator = "-" * 75

    print("\n" + separator)
    print(f"{'Student ID':<12} | {'Name':<22} | {'Branch':<22} | {'CGPA':<5}")
    print(separator)

    for student_id, details in registry.items():
        name = details.get("name", "N/A")
        branch = details.get("branch", "N/A")
        cgpa = details.get("cgpa", 0.0)

        print(f"{student_id:<12} | {name:<22} | {branch:<22} | {cgpa:<5.2f}")

    print(separator + "\n")


# ==============================================================================
# STUDENT TASKS
# ==============================================================================

def add_student_record(registry: Dict[str, Dict[str, Any]]) -> None:
    """Add a new student record with validation."""

    print("\n--- Add New Student ---")

    # Student ID
    student_id = input("Enter Student ID: ").strip()

    if not student_id:
        print("[ERROR] Student ID cannot be empty.")
        return

    if student_id in registry:
        print("[ERROR] Student ID already exists.")
        return

    # Name
    name = input("Enter Student Name: ").strip()

    if not name:
        print("[ERROR] Name cannot be empty.")
        return

    # Branch
    branch = input("Enter Branch: ").strip()

    if not branch:
        print("[ERROR] Branch cannot be empty.")
        return

    # CGPA
    cgpa_input = input("Enter CGPA: ").strip()

    try:
        cgpa = float(cgpa_input)

        if cgpa < 0.0 or cgpa > 10.0:
            print("[ERROR] CGPA must be between 0.0 and 10.0.")
            return

    except ValueError:
        print("[ERROR] CGPA must be a valid number.")
        return

    # Generate email
    first_name = name.split()[0].lower()
    email = f"{first_name}.{student_id.lower()}@university.edu"

    # Add record
    registry[student_id] = {
        "name": name,
        "branch": branch,
        "cgpa": cgpa,
        "email": email
    }

    print("[SUCCESS] Student record added successfully.")
    print(f"Generated Email: {email}")


def search_student_record(registry: Dict[str, Dict[str, Any]]) -> None:
    """Search by student ID or name substring."""

    print("\n--- Search Student Records ---")

    search_term = input("Enter Student ID or Name: ").strip()

    if not search_term:
        print("[ERROR] Search input cannot be empty.")
        return

    found = False

    # Search by ID or name
    for student_id, details in registry.items():

        name = details.get("name", "")

        if (
            student_id.lower() == search_term.lower()
            or search_term.lower() in name.lower()
        ):
            print("\nStudent Found:")
            print(f"Student ID : {student_id}")
            print(f"Name       : {name}")
            print(f"Branch     : {details.get('branch', 'N/A')}")
            print(f"CGPA       : {details.get('cgpa', 0.0):.2f}")
            print(f"Email      : {details.get('email', 'N/A')}")
            print("-" * 40)

            found = True

    if not found:
        print("[INFO] No matching student record found.")


def delete_student_record(registry: Dict[str, Dict[str, Any]]) -> None:
    """Delete a student record after confirmation."""

    print("\n--- Delete Student Record ---")

    student_id = input("Enter Student ID to delete: ").strip()

    if student_id not in registry:
        print("[ERROR] Student ID not found.")
        return

    student = registry[student_id]

    print("\nRecord found:")
    print(f"Name   : {student.get('name', 'N/A')}")
    print(f"Branch : {student.get('branch', 'N/A')}")
    print(f"CGPA   : {student.get('cgpa', 0.0):.2f}")

    confirmation = input("Are you sure you want to delete this record? (y/n): ").strip().lower()

    if confirmation == "y":
        del registry[student_id]
        print("[SUCCESS] Student record deleted successfully.")
    else:
        print("[INFO] Delete operation cancelled.")


def save_records_to_json(
    file_path: str,
    registry: Dict[str, Dict[str, Any]]
) -> None:
    """Save registry safely to JSON file."""

    temp_file = file_path + ".tmp"

    try:
        # Write to temporary file first
        with open(temp_file, "w", encoding="utf-8") as file:
            json.dump(registry, file, indent=2)

            # Flush data to disk
            file.flush()
            os.fsync(file.fileno())

        # Replace original file only after successful write
        os.replace(temp_file, file_path)

        print(f"[SUCCESS] Saved {len(registry)} record(s) to {file_path}.")

    except Exception as err:
        print(f"[ERROR] Failed to save records: {err}")

        # Remove temporary file if something went wrong
        if os.path.exists(temp_file):
            os.remove(temp_file)


def export_to_csv(
    file_path: str,
    registry: Dict[str, Dict[str, Any]]
) -> None:
    """Export all student records to CSV."""

    if not registry:
        print("[INFO] No records available to export.")
        return

    try:
        with open(file_path, "w", newline="", encoding="utf-8") as file:

            fieldnames = [
                "student_id",
                "name",
                "branch",
                "cgpa",
                "email"
            ]

            writer = csv.DictWriter(file, fieldnames=fieldnames)

            writer.writeheader()

            for student_id, details in registry.items():
                writer.writerow({
                    "student_id": student_id,
                    "name": details.get("name", ""),
                    "branch": details.get("branch", ""),
                    "cgpa": details.get("cgpa", ""),
                    "email": details.get("email", "")
                })

        print(f"[SUCCESS] Exported {len(registry)} record(s) to {file_path}.")

    except Exception as err:
        print(f"[ERROR] Failed to export CSV: {err}")


def main_menu() -> None:
    """Main CLI control loop."""

    global STUDENT_REGISTRY

    STUDENT_REGISTRY = load_records_from_json(DATABASE_FILE)

    menu_banner = """
========================================
🎓 STUDENT RECORD MANAGEMENT SYSTEM
========================================
1. View All Records
2. Add Student Record
3. Search Record
4. Delete Record
5. Save Database to File
6. Export Records to CSV (Bonus)
0. Save & Exit
========================================
"""

    while True:

        print(menu_banner)

        choice = input("Enter choice [0-6]: ").strip()

        if choice == "1":
            view_all_records(STUDENT_REGISTRY)

        elif choice == "2":
            add_student_record(STUDENT_REGISTRY)

        elif choice == "3":
            search_student_record(STUDENT_REGISTRY)

        elif choice == "4":
            delete_student_record(STUDENT_REGISTRY)

        elif choice == "5":
            save_records_to_json(DATABASE_FILE, STUDENT_REGISTRY)

        elif choice == "6":
            export_to_csv("students_export.csv", STUDENT_REGISTRY)

        elif choice == "0":
            save_records_to_json(DATABASE_FILE, STUDENT_REGISTRY)
            print("[INFO] Application closed successfully. Good bye!")
            sys.exit(0)

        else:
            print("[WARN] Invalid option selected. Please enter a number between 0 and 6.")


if __name__ == "__main__":
    try:
        main_menu()
    except SystemExit as e:
        # Catch SystemExit to prevent it from being reported as an error in interactive environments
        if e.code != 0:
            raise


[WARN] Database file 'sample_records.json' not found. Starting with empty registry.

🎓 STUDENT RECORD MANAGEMENT SYSTEM
1. View All Records
2. Add Student Record
3. Search Record
4. Delete Record
5. Save Database to File
6. Export Records to CSV (Bonus)
0. Save & Exit

Enter choice [0-6]: 2

--- Add New Student ---
Enter Student ID: 1
Enter Student Name: Khushi
Enter Branch: CSE-AIML
Enter CGPA: 10
[SUCCESS] Student record added successfully.
Generated Email: khushi.1@university.edu

🎓 STUDENT RECORD MANAGEMENT SYSTEM
1. View All Records
2. Add Student Record
3. Search Record
4. Delete Record
5. Save Database to File
6. Export Records to CSV (Bonus)
0. Save & Exit

Enter choice [0-6]: 2

--- Add New Student ---
Enter Student ID: 2
Enter Student Name: Khushi Dua
Enter Branch: Data Science
Enter CGPA: 9.8
[SUCCESS] Student record added successfully.
Generated Email: khushi.2@university.edu

🎓 STUDENT RECORD MANAGEMENT SYSTEM
1. View All Records
2. Add Student Record
3. Search Record
4. D